# Notebook 14 — Dataset Generation

Use Claude to create labelled training and evaluation data for NLP tasks.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Generate sentiment examples

In [ ]:
def generate_samples(label, n=5):
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=512, temperature=0.9,
        messages=[{"role": "user", "content":
            f"Generate {n} short product review sentences that express {label} sentiment. "
            f"Each on its own line. No numbering or quotes."}]
    )
    lines = [l.strip() for l in r.content[0].text.strip().split("\n") if l.strip()]
    return [{"text": l, "label": label} for l in lines[:n]]

dataset = []
for label in ["positive", "negative", "neutral"]:
    samples = generate_samples(label, n=4)
    dataset.extend(samples)
    print(f"{label}: {len(samples)} samples")

print(f"\nTotal: {len(dataset)}")
for item in dataset[:3]:
    print(f"  [{item['label']}] {item['text']}")


## Verify quality with a model-based check

In [ ]:
def check_label(text, expected_label):
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=10, temperature=0,
        messages=[{"role":"user","content":f"Sentiment (positive/negative/neutral):\n{text}\nOne word:"}]
    )
    predicted = r.content[0].text.strip().lower()
    return expected_label.lower() in predicted

mismatches = [item for item in dataset if not check_label(item["text"], item["label"])]
print(f"QA check: {len(dataset)-len(mismatches)}/{len(dataset)} passed")
if mismatches:
    for m in mismatches:
        print(f"  Mismatch: [{m['label']}] {m['text'][:60]}")


## Save to JSON

In [ ]:
out_path = "../datasets/generated_sentiment.json"
with open(out_path, "w") as f:
    json.dump(dataset, f, indent=2)
print(f"Saved {len(dataset)} samples to {out_path}")


## Exercise
Generate a dataset for a different task (e.g., question type classification: factual/opinion/procedural).

In [ ]:
# Your code here
